# Estructura de la base

In [63]:
import pandas as pd

Todas las columnas de la base

In [64]:
df_headers = pd.read_csv('../data/egresos_msp_final.csv', nrows=0)

column_names = df_headers.columns.tolist()
print(column_names)

['area_ubi', 'clase', 'tipo', 'entidad_x', 'sector', 'mes_inv', 'nac_pac', 'nom_pais', 'cod_pais', 'sexo', 'cod_edad', 'edad', 'etnia', 'prov_res', 'area_res', 'anio_ingr', 'mes_ingr', 'dia_ingr', 'fecha_ingr', 'anio_egr', 'mes_egr', 'dia_egr', 'fecha_egr', 'dia_estad', 'con_egrpa', 'esp_egrpa', 'cau_cie10', 'causa3', 'cap221rx', 'cau221rx', 'cau298rx', 'cant_ubi_std_x', 'parr_ubi_std_x', 'cant_res_std', 'parr_res_std', 'ruc', 'eod', 'unicodigo_geosalud', 'nombre_centro_de_salud', 'nivel_de_atencion', 'direccion', 'longps', 'latgps', 'code_provincia_x', 'provincia', 'code_canton', 'canton', 'code_parroquia', 'sri_parroquia', 'parroquia']


Definimos las columnas más importantes

In [65]:
ubicacion = [
    'provincia',
    'canton',
    'parroquia',
    'area_ubi'
]

caracterizacion = [
    'nivel_de_atencion',
    'nombre_centro_de_salud',
    'ruc',
    'tipo',
    'clase'
]

salud = [
    'cau221rx',
    'cau_cie10',
    'dia_estad',
    'fecha_egr',
    'mes_egr',
    'anio_egr',
    'fecha_ingr',
    'mes_ingr',
    'anio_ingr'
]

columns_keep = ubicacion + caracterizacion + salud
df = pd.read_csv('../data/egresos_msp_final.csv', usecols=columns_keep)


# Cau 221 rx

Separamos en 3 columnas: | codigo | nombre | cie |

In [ ]:

df['cod_causa'] = df['cau221rx'].str.extract(r'^(\d+)', expand=False)
df['nombre_causa'] = df['cau221rx'].str.extract(r'(?:^\d+\s+)?(.+?)(?:\s*\(|$)', expand=False)
df['cie_causa'] = df['cau221rx'].str.extract(r'\(([^)]+)\)?', expand=False) 

# Extraer nombre_causa: texto antes del paréntesis O todo el texto después del número
# Extraer cie_causa: manejar tanto (CODE) como (CODE sin cerrar paréntesis

Manejamos error de typo manualmente 

In [67]:
# PASO 1: Quitamos espacios adicionales al inicio y en el medio
df['nombre_causa'] = df['nombre_causa'].str.strip().str.replace(r'\s+', ' ', regex=True)

In [ ]:
# PASO 2: Corregimos manualmente typos
typo_corrections = {
    'Esquizofrenia, trastornos esquizotópicos y trastornos delirantes': 
        'Esquizofrenia, trastornos esquizotípicos y trastornos delirantes',
    
    'Trastorno mental no específicado':
        'Trastorno mental no especificado',
    
    'Síntomas y signos que involucran el conocimiento, la percepción, el estado emoci y la cond':
        'Síntomas y signos que involucran el conocimiento, la percepción, el estado emocional y la conducta',
    
    'Otros efectos y los no específicados de causas externa':
        'Otros efectos y los no especificados de causas externas',
        
    '1Artropatías infecciosas':
        'Artropatías infecciosas',
    
    'Trastornos emocionales y del comportamiento que aparecen habitualmente en la niñez, en la adolesc':
        'Trastornos emocionales y del comportamiento que aparecen habitualmente en la niñez, en la adolescencia',
        
    'Trastornos emocionales y del comportamiento que aparecen habitualmente en la niñez, en la adolescenciaencia':
        'Trastornos emocionales y del comportamiento que aparecen habitualmente en la niñez, en la adolescencia',
        
    'Feto, y recién nacido afectados por factores maternos y complic en el embarazo, del trabajo de parto':
        'Feto, y recién nacido afectados por factores maternos y complicaciones en el embarazo, del trabajo de parto',
    
    'Quemaduras y corrosiones de la superficie externa del cuerpo, específicadas por sitio':
        'Quemaduras y corrosiones de la superficie externa del cuerpo, especificadas por sitio',
    
    'Otros efectos y los no especificados de causas externa':
        'Otros efectos y los no especificados de causas externas',
    
    'Tumor [neoplasias] malignos de sitios mal definidos, secundarios y de sitios no específicados':
        'Tumor [neoplasias] malignos de sitios mal definidos, secundarios y de sitios no especificados',
    
    'Defectos de la coagulación, purpura y otras afecciones hemorrágicas':
        'Defectos de la coagulación, púrpura y otras afecciones hemorrágicas',
    
    'Otros efectos y los no especificados de causas externass':
        'Otros efectos y los no especificados de causas externas',
}

for typo, correct in typo_corrections.items():
    df['nombre_causa'] = df['nombre_causa'].str.replace(typo, correct, regex=False)
    
df.loc[df['nombre_causa'] == 'Artropatías infecciosas', 'cod_causa'] = '129'
df.loc[df['nombre_causa'].str.contains('Personas con riesg', na=False, case=False), 'cod_causa'] = '221'
df.loc[df['cod_causa'] == '221', 'nombre_causa'] = 'Personas con riesgos potenciales para su salud, relacionados con su historia familiar y personal, y algunas condiciones que influyen sobre su estado de salud'
df.loc[df['cod_causa'] == '10', 'cie_causa'] = 'A90-A99'
df.loc[df['cod_causa'] == '215', 'nombre_causa'] = 'Otros efectos y los no especificados de causas externas'

    

Manejemos codigos internos vacios con un mapeo por nombre (normalizado)

In [ ]:
import unicodedata

def remove_accents(text):
    if pd.isna(text):
        return text
    return ''.join(c for c in unicodedata.normalize('NFD', text) 
                   if unicodedata.category(c) != 'Mn')

# Crear versión normalizada (sin acentos, minúsculas, sin espacios adicionales)
df['nombre_causa_norm'] = df['nombre_causa'].apply(remove_accents).str.lower().str.strip().str.replace(r'\s+', ' ', regex=True)

# Crear una búsqueda: para cada nombre_causa_norm único, obtener la PRIMERA cod_causa válida
# Incluir solo filas que ya tengan una cod_causa
lookup = df[df['cod_causa'].notna()].drop_duplicates('nombre_causa_norm').set_index('nombre_causa_norm')['cod_causa'].to_dict()

# Ahora complete el cod_causa faltante según la coincidencia de nombre normalizada
mask = df['cod_causa'].isna()
df.loc[mask, 'cod_causa'] = df.loc[mask, 'nombre_causa_norm'].map(lookup)

RESUMEN : 

In [ ]:
df_resumen_cau221 = df.groupby(['cod_causa', 'nombre_causa', 'cie_causa'], dropna=False).size().reset_index(name='total_count')

In [104]:
# Arreglamos el sort para que sea numerico

df_resumen_cau221['cod_causa_int'] = pd.to_numeric(df_resumen_cau221['cod_causa'], errors='coerce')
df_resumen_cau221 = df_resumen_cau221.sort_values('cod_causa_int').drop('cod_causa_int', axis=1)